# Bayesian optimisation: probability deciding what to try next

**Companion to:** `bayes_29_07_2026.tex`, §*Bayesian optimisation: using probability to
decide what to try next* (`sec:bayes-ml`).

## Historical context

The chapter traces Bayesian optimisation from Harold Kushner's early stochastic-process
approach to noisy optimisation, through Jonas Mockus in the 1970s, to a widely used
modern milestone:

> Jones, D. R., Schonlau, M., & Welch, W. J. (1998). *Efficient Global Optimization of
> Expensive Black-Box Functions*. Journal of Global Optimization, 13(4), 455–492.

and its entry into machine-learning practice:

> Snoek, J., Larochelle, H., & Adams, R. P. (2012). *Practical Bayesian Optimization of
> Machine Learning Algorithms*. Advances in Neural Information Processing Systems
> (NeurIPS) 25.

The chapter also recounts that DeepMind used Bayesian optimisation to tune AlphaGo's
hyperparameters before its 2016 match against Lee Sedol, lifting its self-play win rate
from roughly 50% to 66.5% (Chen et al., 2018, *Bayesian Optimization in AlphaGo*,
arXiv:1812.06855). This notebook runs the same *kind* of workflow at a scale that fits
in a notebook: tuning one real hyperparameter of a real classifier on a real dataset,
using a Gaussian process surrogate (from notebook 5) plus an acquisition function to
decide which setting to try next, benchmarked against random search.

Further reading: Brochu, E., Cora, V. M., & de Freitas, N. (2010), *A Tutorial on
Bayesian Optimization of Expensive Cost Functions*, arXiv:1012.2599; Shahriari, B. et
al. (2016), *Taking the Human Out of the Loop: A Review of Bayesian Optimization*,
Proceedings of the IEEE.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from utils.plotting import set_style, save_fig, PALETTE

set_style()
NB_DIR = pathlib.Path.cwd()
rng = np.random.default_rng(5)

## Problem & data (real)

**Real dataset**: the scikit-learn breast-cancer diagnostic dataset (569 samples, 30
features). **Real, expensive black-box objective**: 5-fold cross-validated accuracy of
an RBF-kernel support vector classifier, as a function of its regularisation strength
$C$. We search over $\log_{10}C \in [-3, 3]$ — evaluating this function means actually
training and cross-validating an SVM, which is exactly the "training a model may itself
be extremely expensive" situation the chapter describes (here cheap enough to run many
times for the demo, but the same principle scales to settings where each evaluation
takes hours).

In [ ]:
X_data, y_data = load_breast_cancer(return_X_y=True)


def objective(log10_C):
    '''Real black-box objective: 5-fold CV accuracy of an RBF-SVM at this C.'''
    C = 10.0 ** log10_C
    model = SVC(C=C, kernel="rbf", gamma="scale")
    scores = cross_val_score(model, X_data, y_data, cv=5)
    return scores.mean()


BOUNDS = (-3.0, 3.0)

# a dense grid evaluation, purely so we know the true optimum for scoring the search
# methods afterwards -- neither BO nor random search is allowed to see this grid.
grid = np.linspace(*BOUNDS, 121)
grid_scores = np.array([objective(g) for g in grid])
true_best = grid_scores.max()
print(f"(Reference only) best CV accuracy on the dense grid: {true_best:.4f} "
      f"at log10(C) = {grid[grid_scores.argmax()]:.2f}")

## Model: Gaussian process surrogate + expected improvement

At each step we (1) fit a GP to the points evaluated so far, exactly as in notebook 5,
(2) compute the **expected improvement (EI)** acquisition function over the search
space, which trades off exploiting points near the current best against exploring
points where the GP is still uncertain, and (3) evaluate the real objective at the
point that maximises EI.

$$
\mathrm{EI}(x) = (\mu(x) - f^\star)\,\Phi(Z) + \sigma(x)\,\phi(Z),
\qquad Z = \frac{\mu(x) - f^\star}{\sigma(x)},
$$

where $f^\star$ is the best value observed so far, and $\Phi,\phi$ are the standard
normal CDF/PDF (maximisation form, since we're maximising accuracy).

In [ ]:
def expected_improvement(mu, sigma, f_best, xi=0.01):
    sigma = np.maximum(sigma, 1e-9)
    Z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)


def bayesian_optimisation(objective, bounds, n_init, n_iter, rng, xi=0.01):
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(1e-4)
    X_obs = rng.uniform(*bounds, size=n_init).reshape(-1, 1)
    y_obs = np.array([objective(x[0]) for x in X_obs])

    candidates = np.linspace(*bounds, 400).reshape(-1, 1)
    best_so_far = [y_obs.max()]

    for _ in range(n_iter):
        gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                       n_restarts_optimizer=2, random_state=0)
        gp.fit(X_obs, y_obs)

        mu, sigma = gp.predict(candidates, return_std=True)
        ei = expected_improvement(mu, sigma, y_obs.max(), xi=xi)
        x_next = candidates[np.argmax(ei)]

        y_next = objective(x_next[0])
        X_obs = np.vstack([X_obs, x_next])
        y_obs = np.append(y_obs, y_next)
        best_so_far.append(y_obs.max())

    return X_obs, y_obs, np.array(best_so_far), gp


def random_search(objective, bounds, n_evals, rng):
    X_obs = rng.uniform(*bounds, size=n_evals).reshape(-1, 1)
    y_obs = np.array([objective(x[0]) for x in X_obs])
    best_so_far = np.maximum.accumulate(y_obs)
    return X_obs, y_obs, best_so_far

## Run: Bayesian optimisation vs. random search, same evaluation budget

We repeat both methods several times (different random starting points) so the
comparison isn't a fluke of one lucky/unlucky run, and compare **simple regret** — the
gap between the best found so far and the true optimum — as a function of the number of
real objective evaluations.

In [ ]:
N_INIT, N_ITER, N_REPEATS = 3, 12, 6
TOTAL_EVALS = N_INIT + N_ITER

bo_curves = np.empty((N_REPEATS, TOTAL_EVALS))
rs_curves = np.empty((N_REPEATS, TOTAL_EVALS))

last_gp = None
last_X_obs, last_y_obs = None, None

for r in range(N_REPEATS):
    run_rng = np.random.default_rng(100 + r)
    X_obs, y_obs, best_so_far, gp = bayesian_optimisation(
        objective, BOUNDS, N_INIT, N_ITER, run_rng
    )
    bo_curves[r] = best_so_far
    if r == 0:
        last_gp, last_X_obs, last_y_obs = gp, X_obs, y_obs

    run_rng2 = np.random.default_rng(200 + r)
    _, _, rs_best = random_search(objective, BOUNDS, TOTAL_EVALS, run_rng2)
    rs_curves[r] = rs_best

bo_regret = true_best - bo_curves
rs_regret = true_best - rs_curves

## Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# --- Left: GP surrogate + acquisition snapshot from the first BO run ---
candidates = np.linspace(*BOUNDS, 400).reshape(-1, 1)
mu, sigma = last_gp.predict(candidates, return_std=True)
ei = expected_improvement(mu, sigma, last_y_obs.max())

ax = axes[0]
ax.plot(grid, grid_scores, color=PALETTE["true"], lw=1.5, label="true objective (reference grid)")
ax.plot(candidates, mu, color=PALETTE["approx"], lw=1.5, label="GP posterior mean")
ax.fill_between(candidates.ravel(), mu - 2 * sigma, mu + 2 * sigma,
                 color=PALETTE["approx"], alpha=0.2, label="GP 95% band")
ax.scatter(last_X_obs, last_y_obs, color="black", zorder=5, s=30, label="evaluated points")
ax.set_title("GP surrogate after Bayesian optimisation")
ax.set_xlabel(r"$\log_{10} C$")
ax.set_ylabel("5-fold CV accuracy")
ax.legend(fontsize=8, loc="lower center")

ax2 = ax.twinx()
ax2.plot(candidates, ei, color=PALETTE["accent"], lw=1.0, ls="--", alpha=0.7)
ax2.set_ylabel("expected improvement", color=PALETTE["accent"])
ax2.tick_params(axis="y", colors=PALETTE["accent"])

# --- Right: simple regret, BO vs random search ---
evals = np.arange(1, TOTAL_EVALS + 1)
ax = axes[1]
for curves, color, label in [(bo_regret, PALETTE["approx"], "Bayesian optimisation"),
                              (rs_regret, PALETTE["muted"], "random search")]:
    mean_regret = curves.mean(axis=0)
    std_regret = curves.std(axis=0)
    ax.plot(evals, mean_regret, color=color, lw=2, label=label)
    ax.fill_between(evals, mean_regret - std_regret, mean_regret + std_regret,
                     color=color, alpha=0.2)
ax.set_title(f"Simple regret vs. evaluation budget ({N_REPEATS} repeats)")
ax.set_xlabel("number of real objective evaluations")
ax.set_ylabel("regret (true best − best found so far)")
ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, "bayesopt_vs_random_search", NB_DIR)
plt.show()

print(f"Mean final regret -- Bayesian optimisation: {bo_regret[:, -1].mean():.5f}")
print(f"Mean final regret -- random search:         {rs_regret[:, -1].mean():.5f}")

## Takeaways

- The GP surrogate turns a handful of real, expensive SVM cross-validation runs into a
  smooth belief over the *entire* accuracy-vs-$C$ curve, with honest uncertainty away
  from evaluated points — visibly wider where fewer points have been tried.
  Expected improvement uses exactly that uncertainty to decide where to look next,
  rather than searching blindly.
- Averaged over repeats, Bayesian optimisation reaches a lower simple regret than
  random search for the same number of real evaluations — the practical payoff the
  chapter attributes to Snoek, Larochelle & Adams' 2012 result, and to DeepMind's use
  of Bayesian optimisation on AlphaGo's hyperparameters: **probability deciding which
  expensive experiment to run next**, rather than an exhaustive or random sweep.
- This closes the loop on the six notebooks: the same GP machinery from notebook 5,
  built on the same Bayes'-theorem machinery from notebook 1, applied here not to
  describe a fixed dataset but to *choose what data to collect next* — inference and
  decision-making, the two threads the chapter follows from Bayes and Laplace all the
  way to modern machine learning.